# Credit Risk Analysis: Mortgage Loan Defaults
## ETL Pipeline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path # ensures compatibility of file path strings across Windows/Mac/Linux
import zipfile # for programmatically unzipping big data files
import os # to set working directory

In [4]:
# Set working directory
os.chdir("C:/Projects-Code-Certificates/Credit-Risk-Mortgages")

In [5]:
# column names for origination
orig_cols = [
    "credit_score",                      # 1
    "first_payment_date",                # 2 (YYYYMM)
    "first_time_homebuyer_flag",         # 3
    "maturity_date",                     # 4 (YYYYMM)
    "msa",                               # 5
    "mi_percent",                        # 6
    "num_units",                         # 7
    "occupancy_status",                  # 8
    "orig_cltv",                         # 9
    "orig_dti",                          # 10
    "orig_upb",                          # 11
    "orig_ltv",                          # 12
    "orig_interest_rate",                # 13
    "channel",                           # 14
    "ppm_flag",                          # 15
    "amortization_type",                 # 16
    "property_state",                    # 17
    "property_type",                     # 18
    "postal_code",                       # 19 (keep as string to preserve leading zeros)
    "loan_sequence_number",              # 20 (loan ID)
    "loan_purpose",                      # 21
    "orig_loan_term",                    # 22
    "num_borrowers",                     # 23
    "seller_name",                       # 24
    "servicer_name",                     # 25
    "super_conforming_flag",             # 26
    "pre_harp_loan_id",                  # 27
    "program_indicator",                 # 28
    "harp_indicator",                    # 29
    "property_valuation_method",         # 30
    "interest_only_indicator",           # 31
    "mi_cancellation_indicator",         # 32
]

# data types dictionary for origination
orig_dtypes = {
    "credit_score": "Int64",
    "msa": "Int64",
    "mi_percent": "Int64",
    "num_units": "Int64",
    "orig_cltv": "Int64",
    "orig_dti": "Int64",
    "orig_upb": "Int64",
    "orig_ltv": "Int64",
    "orig_interest_rate": "float64",
    "postal_code": "string",
    "loan_sequence_number": "string",
    "pre_harp_loan_id": "string",
    "seller_name": "string",
    "servicer_name": "string",
    "first_time_homebuyer_flag": "string",
    "occupancy_status": "string",
    "channel": "string",
    "ppm_flag": "string",
    "amortization_type": "string",
    "property_state": "string",
    "property_type": "string",
    "loan_purpose": "string",
    "super_conforming_flag": "string",
    "program_indicator": "string",
    "harp_indicator": "string",
    "property_valuation_method": "string",
    "interest_only_indicator": "string",
    "mi_cancellation_indicator": "string",
    "orig_loan_term": "Int64",
    "num_borrowers": "Int64",
} 


# column names for monthly performance
perf_cols = [
    "loan_sequence_number",                 # 1
    "monthly_reporting_period",             # 2 (YYYYMM)
    "current_actual_upb",                   # 3
    "current_loan_delinquency_status",      # 4
    "loan_age",                             # 5
    "remaining_months_to_legal_maturity",   # 6
    "defect_settlement_date",               # 7 (YYYYMM)
    "modification_flag",                    # 8
    "zero_balance_code",                    # 9
    "zero_balance_effective_date",          # 10 (YYYYMM)
    "current_interest_rate",                # 11
    "current_deferred_upb",                 # 12
    "ddlpi",                                # 13 (YYYYMM) Due Date of Last Paid Installment
    "mi_recoveries",                        # 14
    "net_sales_proceeds",                   # 15 (alpha-numeric)
    "non_mi_recoveries",                    # 16
    "expenses",                             # 17
    "legal_costs",                          # 18
    "maintenance_and_preservation_costs",   # 19
    "taxes_and_insurance",                  # 20
    "miscellaneous_expenses",               # 21
    "actual_loss_calculation",              # 22
    "modification_cost",                    # 23
    "step_modification_flag",               # 24
    "deferred_payment_plan",                # 25
    "estimated_ltv_eltv",                   # 26
    "zero_balance_removal_upb",             # 27
    "delinquent_accrued_interest",          # 28
    "delinquency_due_to_disaster",          # 29
    "borrower_assistance_status_code",      # 30
    "current_month_modification_cost",      # 31
    "interest_bearing_upb"                  # 32
]

# data types names for monthly performance
perf_dtypes = {
        "loan_sequence_number": "string",
        "current_loan_delinquency_status": "string",
        "modification_flag": "string",
        "step_modification_flag": "string",
        "deferred_payment_plan": "string",
        "delinquency_due_to_disaster": "string",
        "borrower_assistance_status_code": "string",
        "net_sales_proceeds": "string", # keep zero_balance_code as string to preserve leading zeros like "01"
        "zero_balance_code": "string",
    }

# numeric columns for monthly performance data
num_cols = [
    "current_actual_upb",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_interest_rate",
    "current_deferred_upb",
    "mi_recoveries",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss_calculation",
    "modification_cost",
    "estimated_ltv_eltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "current_month_modification_cost",
    "interest_bearing_upb",
]

# columns needed for Markov chain modeling
dtmc_perf_cols = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "deliq_num",
    "loan_age"
]

# columns needed for regression (regression modeling done in separate document)
regres_perf_cols = dtmc_perf_cols + [
    "current_actual_upb",
    "current_interest_rate",
]
regres_orig_cols = [
    "quarter", 
    "loan_sequence_number",
    "credit_score",
    "orig_ltv",
    "orig_dti",
    "orig_upb",
    "orig_interest_rate",
    "orig_loan_term",
    "occupancy_status",
    "property_state", 
    "loan_purpose", 
    "amortization_type", 
    "ppm_flag"
]

# date columns
perf_date_cols = ["monthly_reporting_period", "defect_settlement_date", "zero_balance_effective_date", "ddlpi"]
orig_date_cols = ["first_payment_date", "maturity_date"]

# custom defined states of deliquency
states = ["Prepaid", "Current", "30 DPD", "60 DPD", "Default"]

# performance and origination columns to be used post-cleaning
perf_cols_pc = ["loan_sequence_number", 
                "monthly_reporting_period", 
                "state", 
                "loan_age", 
                "current_interest_rate", 
                "current_actual_upb", 
                "deliq_num"]

orig_cols_pc = ["loan_sequence_number", 
                "amortization_type", 
                "quarter", 
                "ppm_flag", 
                "orig_ltv", 
                "orig_dti", 
                "credit_score", 
                "orig_loan_term", 
                "orig_interest_rate", 
                "orig_upb", 
                "property_state", 
                "occupancy_status", 
                "loan_purpose"]


In [6]:
# function to read in origination dataset
def read_orig(file):
    return pd.read_csv(
        file,
        sep="|",
        header=None,
        names=orig_cols,
        dtype=orig_dtypes,
        na_values=["", " ", "NA", "N/A"],
        keep_default_na=True,
        low_memory=False
    )

# function to read in monthly performance dataset
def read_perf(file, chunksize=None):
    if chunksize == None:
        return pd.read_csv(
            file,
            sep="|",
            header=None,
            names=perf_cols,
            dtype=perf_dtypes,
            na_values=["", " ", "NA", "N/A"], 
            keep_default_na=True, 
            low_memory=False
            )
    else:
        return pd.read_csv(
            file,
            sep="|",
            header=None,
            names=perf_cols,
            dtype=perf_dtypes,
            na_values=["", " ", "NA", "N/A"], 
            keep_default_na=True, 
            chunksize=chunksize, 
            low_memory=False
        )

# function to select relevant loans from monthly performance data
def filter_perf(file, loan_ids, chunksize=1_000_000):
    out = []
    for chunk in read_perf(file=file, chunksize=chunksize):
        chunk = chunk[chunk["loan_sequence_number"].isin(loan_ids)]
        if not chunk.empty:
            out.append(chunk)
    if len(out) > 0:
        return pd.concat(out, ignore_index=True)
    else:
        return pd.DataFrame(columns=perf_cols)
    
# map deliquency status to defined delinquency states
def deliq_state(x):
    if pd.isna(x):
        return "Other"
    if x == 0:
        return "Current"
    elif x == 1:
        return "30 DPD"
    elif x == 2:
        return "60 DPD"
    elif x >= 3:
        return "Default"
    else:
        return "Other"
    
# define prepaid state
def prepaid_state(row):
    zb = row["zero_balance_code"]
    if zb == "NA":
        return row["state"]
    elif zb == "01":
        return "Prepaid"
    elif zb == "03":
        return "Default"

In [7]:
# function to clean and reduce size of data for a sample from one year's vintage
def clean_data(orig, perf):

    # clean deliquency status column
    bad_ids = set(perf.loc[perf['current_loan_delinquency_status'].isin(['', 'RA']), "loan_sequence_number"].unique().tolist())
    perf = perf[~perf["loan_sequence_number"].isin(bad_ids)].copy()
    perf['deliq_num'] = perf['current_loan_delinquency_status'].astype(int)

    # format dates
    for c in perf_date_cols:
        perf[c] = pd.to_datetime(perf[c], format="%Y%m", errors="coerce")
    for c in orig_date_cols:
        orig[c] = pd.to_datetime(orig[c], format="%Y%m", errors="coerce")

    # convert selected columns to floats
    for c in num_cols:
        perf[c] = pd.to_numeric(perf[c], errors="coerce")

    # coerce NA values for zero balance codes to strings
    perf['zero_balance_code'] = perf['zero_balance_code'].fillna("NA")

    # drop loans with very rare zero balance codes
    bad_ids = set(perf.loc[~perf['zero_balance_code'].isin(['NA', '01', '03']), "loan_sequence_number"].unique().tolist())
    perf = perf[~perf["loan_sequence_number"].isin(bad_ids)].copy()

    # apply definitions of delinquent states (default, prepaid, etc.)
    perf["state"] = perf["deliq_num"].apply(deliq_state)
    perf["state"] = perf.apply(prepaid_state, axis=1)

    # subset data to ensure only well-defined states are included
    bad_ids = set(perf.loc[~perf["state"].isin(states), "loan_sequence_number"].unique().tolist())
    perf = perf[~perf["loan_sequence_number"].isin(bad_ids)].copy()

    # select subset of columns
    perf = perf[perf_cols_pc].copy()
    orig = orig[orig_cols_pc].copy()

    return orig, perf


In [8]:
seed = 10 # seeds for simple random sampling (for reproducibility)
y_cohort_size = 100_000 # number of loans to sample from each year (300K total sample size pre-cleaning)

years = ["2013", "2016", "2020"] # sample years
quarters = ["Q1", "Q2", "Q3", "Q4"]

project_root = Path.cwd()
base_dir =  Path("data/raw")
out_dir = Path("data/processed")

(out_dir / "DTMC").mkdir(parents=True, exist_ok=True)
(out_dir / "Regression").mkdir(parents=True, exist_ok=True)

# initialize list of dataframes (`lodf`) for origination and performance data
lodf_orig = [] 
lodf_perf = []

In [9]:
# unzip raw data (if needed)
for yr in years:

    zip_path = base_dir / f"historical_data_{yr}.zip"
    target_dir = base_dir / f"historical_data_{yr}"

    if not target_dir.exists() or not any(target_dir.iterdir()):
        print(f"Unzipping {zip_path}")
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(target_dir)
    else:
        print(f"Skipping unzip for {yr}; already done")

    for qzip in target_dir.glob("*.zip"):
        qdir = target_dir / qzip.stem   # e.g., historical_data_2013Q1

        if not qdir.exists() or not any(qdir.iterdir()):
            print(f"\t Unzipping quarter {qdir.name}")
            qdir.mkdir(exist_ok=True)
            with zipfile.ZipFile(qzip, "r") as zf:
                zf.extractall(qdir)
        else:
            print(f"\t Quarter already unzipped for {qdir.name}")

Unzipping data\raw\historical_data_2013.zip
	 Unzipping quarter historical_data_2013Q1
	 Unzipping quarter historical_data_2013Q2
	 Unzipping quarter historical_data_2013Q3
	 Unzipping quarter historical_data_2013Q4
Unzipping data\raw\historical_data_2016.zip
	 Unzipping quarter historical_data_2016Q1
	 Unzipping quarter historical_data_2016Q2
	 Unzipping quarter historical_data_2016Q3
	 Unzipping quarter historical_data_2016Q4
Unzipping data\raw\historical_data_2020.zip
	 Unzipping quarter historical_data_2020Q1
	 Unzipping quarter historical_data_2020Q2
	 Unzipping quarter historical_data_2020Q3
	 Unzipping quarter historical_data_2020Q4


In [10]:
# process data
for yr in years:

    y_str = "historical_data_" + yr
    y_path = base_dir / y_str

    y_lodf_orig = [] 
    y_lodf_perf = []

    # sample from each quarter one-by-one to save memory
    for qtr in quarters:

        qy_str = y_str + qtr

        qy_path = y_path / qy_str

        qy_orig_path = qy_path / f"{qy_str}.txt"
        qy_perf_path = qy_path / f"historical_data_time_{yr}{qtr}.txt"

        qy_orig_df = read_orig(qy_orig_path).sample(n=y_cohort_size//4, random_state=seed) 
        
        qy_orig_df['quarter'] = qtr
        qy_orig_df["vintage_year"] = int(yr)

        qy_ids = set(qy_orig_df['loan_sequence_number'].unique().tolist())
        qy_perf_df = filter_perf(qy_perf_path, qy_ids, chunksize=250_000)

        y_lodf_orig.append(qy_orig_df)
        y_lodf_perf.append(qy_perf_df)

    # aggregate quarterly data for this year
    y_orig_df = pd.concat(y_lodf_orig, ignore_index=True)
    
    # aggregate monthly performance data for this year 
    y_perf_df = pd.concat(y_lodf_perf, ignore_index=True)

    size_bytes = y_perf_df.memory_usage(deep=True).sum()
    print(f"Pre-Clean Size of {yr} Performance Data: {size_bytes / 1024**3:.2f} GB")
    
    # clean origination and performance data for this year (doing it now to save memory)
    y_orig_df, y_perf_df = clean_data(y_orig_df, y_perf_df)
    size_bytes = y_perf_df.memory_usage(deep=True).sum()
    print(f"Post-Clean Size of {yr} Performance Data: {size_bytes / 1024**3:.2f} GB\n")

    lodf_orig.append(y_orig_df)
    lodf_perf.append(y_perf_df)

    
# aggregate samples from each year
orig_df = pd.concat(lodf_orig, ignore_index=True)
perf_df = pd.concat(lodf_perf, ignore_index=True)


# export processed data for future use
dtmc_data = perf_df[dtmc_perf_cols]
dtmc_data.to_csv(out_dir / Path('DTMC/dtmc_data.csv'))
regres_orig = orig_df[regres_orig_cols]
regres_orig.to_csv(out_dir / Path('Regression/orig_data.csv'))
perf_df.to_csv(out_dir / Path('Regression/perf_data.csv'))


Pre-Clean Size of 2013 Performance Data: 5.26 GB
Post-Clean Size of 2013 Performance Data: 1.25 GB

Pre-Clean Size of 2016 Performance Data: 4.16 GB
Post-Clean Size of 2016 Performance Data: 1.00 GB

Pre-Clean Size of 2020 Performance Data: 2.87 GB
Post-Clean Size of 2020 Performance Data: 0.69 GB

